# 1D FWI implementation via ODIL

In this notebook, I will extend what I have learnt of ODIL from reproducing Figure 1 in the [`/reproduction`](./../reproduction/) notebooks. Our task will be to jointly optimise for an amplitude field and a wavespeed field using the ODIL framework. This will involve introducing a source term, the concept of 'receievers' (discrete observation points), and a data misfit.

To start, we can update our `Wavefield` class to include a wavespeed model in its data vector. It is important we keep the wavespeed and ampltiude data together. If we do not perform a joint optimisation and instead optimise the two field separately, the wave equation solve will be chasing an expired wavespeed model. 

In [4]:
from dataclasses import dataclass, field
from typing import Tuple
import numpy as np
import matplotlib.pyplot as plt

In [13]:
@dataclass
class Wavefield:
    # spatial data
    x_0: float = -1.0
    x_I: float = 1.0
    I: int = 25
    L: float = field(init=False)
    dx: float = field(init=False)

    # temporal data
    t_0: float = 0.0
    t_N: float = 1.0
    N: int = 25
    T: float = field(init=False)
    dt: float = field(init=False)

    # actual data
    init_amplitude: np.ndarray | None = None  # data to initialise with
    init_wavespeed: np.ndarray | None = None
    wavespeed_idx: float = field(init=False) # where wavespeed starts in internal data

    _data: np.ndarray = field(init=False)  # internal data
    _shape: Tuple[int, int] = field(init=False)  # length of domain object

    def __post_init__(self):
        # compute lengths and spacings
        self.L = self.x_I - self.x_0
        self.dx = self.L / (self.I - 1)

        self.T = self.t_N - self.t_0
        self.dt = self.T / (self.N - 1)

        self._shape = (self.N, self.I)  # t, x

        self.wavespeed_idx = self.N * self.I 

        # receieve initial values or init arrays
        amplitude = (
            np.zeros(shape=(self._shape), dtype=float).flatten() if self.init_amplitude is None 
            else np.asarray(self.init_amplitude, dtype=float).flatten()
            )
        wavespeed = (np.ones(shape=(self.I)) if self.init_wavespeed is None 
                    else np.asarray(self.init_wavespeed, dtype=float).flatten()
                    )

        # concatenate the two
        # note: wavespeed starts at NxI
        self._data = np.concatenate((amplitude, wavespeed))

    # convention: space index i, time index n
    def __getitem__(self, key):
        i, n = key
        return self._data[i + self.I * n]

    def __setitem__(self, key, value):
        i, n = key
        self._data[i + self.I * n] = value

    def __sub__(self, other):
        return self._data - other._data

    def __pow__(self, other):
        return self._data**other

    @property
    def data(self):
        return self._data
    
    @property
    def amplitude(self):
        return self._data[:self.wavespeed_idx]
    
    @property
    def wavespeed(self):
        return self._data[self.wavespeed_idx:]

    @property
    def shape(self):
        return self._shape

    def show(self, title: str = "Exact solution") -> None:
        fig, ax = plt.subplots(figsize=(8, 5))
        umax = np.abs(self.amplitude).max()
        im = ax.imshow(
            self.amplitude.reshape(self.N, self.I),
            extent=(self.x_0, self.x_I, self.t_N, self.t_0),  # [xmin, xmax, tmax, tmin]
            cmap="RdBu_r",
            vmin=-umax,
            vmax=umax,
            aspect=4,
        )

        plt.colorbar(im, ax=ax, label="u(x, t)")
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(title)
        plt.tight_layout()
        plt.show()

In [14]:
u = Wavefield(init_wavespeed=np.random.rand(25, 25), init_amplitude=np.random.rand(25, 25))

Now, we can start to work on our loss formulation. There are a few things we need to do:

- Previously, we used `c=1`. Now, we need to update our residual to use the wavespeed model at the interior points.
- We used analytical BCs, which won't be available in an actual inversion. We need to figure out some new BCs to enforce, such as PMLs, and add a source term that will drive the forward solve.
- Further to the above, we will need to figure out where to inject our source and where to observe the wavefield, i.e., where to place our receivers. This controls the ill-posedness of the problem and will be important to get right.
- We need to add a data misfit. This will involve solving the forward problem with some target model `c_true` as the initial value to create an observed data set, then adding this in as a residual term.
- We will likely need to regularise the components, since having sparse observations will introduce a significant underdeterminism.

Our updated PDE loss will become

$$\mathcal{R}_\text{PDE} = \frac {u^{n+1}_i -2u^n_i + u^{n-1}_i} {\Delta t^2} - (c_\text{interior})^2 \frac {u^n_{i+1} -2u^n_i + u^{n}_{i+1}} {\Delta x^2}$$

In [17]:
import torch

In [ ]:
class DiscretePDELoss:
    def __init__(self, u: Wavefield):
        self.u = u
        self.history = {
            "R_pde": [],
            "R_ic": [],
            "R_vel": [],
            "R_left": [],
            "R_right": [],
            "L": [],
        }

    # evaluate full loss and gradient
    def evaluate(self, params: np.ndarray) -> Tuple[float, np.ndarray]:
        p = torch.tensor(params, requires_grad=True, dtype=torch.float64)
        r = self._residuals(p)
        L = (r**2).sum()
        L.backward()
        # p.grad can be None in some static-analysis scenarios
        grad = p.grad if p.grad is not None else torch.zeros_like(p)
        return L.item(), grad.numpy()

    def _residuals(self, p_flat: torch.Tensor) -> torch.Tensor:
        r = self._compute_residuals(p_flat)
        self._log(r)
        return r

    def _compute_residuals(self, p_flat: torch.Tensor) -> torch.Tensor:
        p = p_flat.reshape(self.u.N, self.u.I)

        # PDE residual
        utt = (p[2:, 1:-1] - 2 * p[1:-1, 1:-1] + p[:-2, 1:-1]) / self.u.dt**2
        uxx = (p[1:-1, 2:] - 2 * p[1:-1, 1:-1] + p[1:-1, :-2]) / self.u.dx**2

        c_int = self.u.wavespeed[1:-1] # extract interior points of the wavespeed model
        
        R_pde = (utt - c_int**2 * uxx).ravel()

        # boundary conditions
        xs = torch.linspace(self.u.x_0, self.u.x_I, self.u.I, dtype=torch.float64)
        ts = torch.linspace(self.u.t_0, self.u.t_N, self.u.N, dtype=torch.float64)

        # allocate for BCs
        g_ic = torch.zeros(self.u.I, dtype=torch.float64)
        g_vel = torch.zeros(self.u.I, dtype=torch.float64)
        g_left = torch.zeros(self.u.N, dtype=torch.float64)
        g_right = torch.zeros(self.u.N, dtype=torch.float64)

        # evaluate exact soln
        for k in range(1, 6):
            kpi = k * torch.pi
            g_ic += 0.1 * (torch.cos((xs + 0.5) * kpi) + torch.cos((xs - 0.5) * kpi))
            g_vel += (
                0.1 * kpi * (torch.sin((xs + 0.5) * kpi) - torch.sin((xs - 0.5) * kpi))
            )  # u_t(x,0) = 0
            g_left += 0.1 * (
                torch.cos((self.u.x_0 - ts + 0.5) * kpi)
                + torch.cos((self.u.x_0 + ts - 0.5) * kpi)
            )
            g_right += 0.1 * (
                torch.cos((self.u.x_I - ts + 0.5) * kpi)
                + torch.cos((self.u.x_I + ts - 0.5) * kpi)
            )

        # compute residuals
        R_ic = p[0, :] - g_ic
        R_vel = (p[1, :] - p[0, :]) / self.u.dt - g_vel
        R_left = p[:, 0] - g_left
        R_right = p[:, -1] - g_right

        return torch.cat([R_pde, R_ic, R_vel, R_left, R_right])

    def _log(self, r: torch.Tensor) -> None:
        n_pde = (self.u.N - 2) * (self.u.I - 2)  # only get interiors
        n_ic = self.u.I
        n_vel = self.u.I
        n_left = self.u.N
        n_right = self.u.N

        idx = 0
        self.history["R_pde"].append(r[idx : idx + n_pde].norm().item())
        idx += n_pde
        self.history["R_ic"].append(r[idx : idx + n_ic].norm().item())
        idx += n_ic
        self.history["R_vel"].append(r[idx : idx + n_vel].norm().item())
        idx += n_vel
        self.history["R_left"].append(r[idx : idx + n_left].norm().item())
        idx += n_left
        self.history["R_right"].append(r[idx : idx + n_right].norm().item())
        self.history["L"].append((r**2).sum().item())

    # plot residual history
    def plot_history(self, title: str = "Residual history"):
        if len(self.history["R_pde"]) == 0:
            raise BufferError(
                "Detected history of length 0. Please optimise before trying to plot."
            )
        fig, axs = plt.subplots(2, 3, figsize=(12, 8))

        titles = [
            r"$\|\mathcal{R}_\mathrm{PDE}\|^2$",
            r"$\|\mathcal{R}_\mathrm{ic}\|$",
            r"$\|\mathcal{R}_\mathrm{vel}\|$",
            r"$\|\mathcal{R}_\mathrm{left}\|$",
            r"$\|\mathcal{R}_\mathrm{right}\|$",
            r"$L(u_i^n)$",
        ]

        for ax, t, key in zip(axs.ravel(), titles, self.history.keys()):
            data = self.history[key]
            if not data:
                ax.set_visible(False)
                continue
            ax.plot(np.arange(len(data)), data)  # iters, data
            ax.set_xlabel("Function evaluation")
            ax.set_title(t)
            ax.set_yscale("log")

        fig.suptitle(title)
        plt.tight_layout()
        plt.show()
